In [ ]:
import os
import shutil
import kagglehub

# Download latest version
path = kagglehub.dataset_download("simhadrisadaram/mimic-cxr-dataset")

print("Path to dataset files:", path)

# Step 2: Create output folder
output_dir = "MIMIC-CXR_records_images"
os.makedirs(output_dir, exist_ok=True)

# Supported image formats
image_extensions = (".png", ".jpg", ".jpeg", ".bmp", ".tiff")

# Limit
max_images = 1200

# Step 3: Walk through dataset and collect images
image_count = 0

for root, dirs, files in os.walk(path):
    for file in files:
        if file.lower().endswith(image_extensions):
            source_path = os.path.join(root, file)

            # Avoid filename conflicts
            new_filename = f"img_{image_count}_{file}"
            dest_path = os.path.join(output_dir, new_filename)

            shutil.copy2(source_path, dest_path)
            image_count += 1

            # Stop when limit reached
            if image_count >= max_images:
                break
    if image_count >= max_images:
        break

print(f"✅ Total images extracted: {image_count}")
print(f"📁 Images saved in: {output_dir}")

Remove duplicates

In [ ]:
import os
import hashlib

image_dir = "MIMIC-CXR_records_images"

def get_file_hash(filepath):
    hasher = hashlib.md5()
    with open(filepath, 'rb') as f:
        buf = f.read()
        hasher.update(buf)
    return hasher.hexdigest()

hash_dict = {}
duplicates = []

for filename in os.listdir(image_dir):
    path = os.path.join(image_dir, filename)
    
    try:
        file_hash = get_file_hash(path)
        
        if file_hash in hash_dict:
            duplicates.append(path)
        else:
            hash_dict[file_hash] = path
    except:
        print(f"Error reading {filename}")

# Remove duplicates
for dup in duplicates:
    os.remove(dup)

print(f"✅ Removed {len(duplicates)} exact duplicates")

Remove Corrupt / Invalid Images

In [ ]:
from PIL import Image

removed = 0

for filename in os.listdir(image_dir):
    path = os.path.join(image_dir, filename)
    
    try:
        with Image.open(path) as img:
            img.verify()  # check corruption
    except:
        os.remove(path)
        removed += 1

print(f"✅ Removed {removed} corrupt images")

remove low-quality images

In [ ]:
from PIL import Image

min_width = 100
min_height = 100

removed = 0

for filename in os.listdir(image_dir):
    path = os.path.join(image_dir, filename)
    
    try:
        with Image.open(path) as img:
            width, height = img.size
            
            if width < min_width or height < min_height:
                os.remove(path)
                removed += 1
    except:
        pass

print(f"✅ Removed {removed} low-resolution images")

Final Count Check

In [ ]:
print("📊 Final image count:", len(os.listdir(image_dir)))